In [1]:
import pyspark

In [2]:
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession.builder.appName('Missing').getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/16 13:09:46 WARN Utils: Your hostname, SAJINs-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.4 instead (on interface en0)
26/07/16 13:09:46 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/16 13:09:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
spark

In [5]:
training = spark.read.csv('employee_dataset.csv',header = True,inferSchema = True)

In [31]:
training = training.na.drop()

In [32]:
training.show()

+---------+----+----------+-------+
|     Name| Age|Experience| Salary|
+---------+----+----------+-------+
|    Krish|31.0|      10.0|30000.0|
|Sudhanshu|30.0|       8.0|25000.0|
|    Sunny|29.0|       4.0|20000.0|
|     Paul|24.0|       3.0|20000.0|
|   Harsha|21.0|       1.0|15000.0|
|  Shubham|23.0|       2.0|18000.0|
+---------+----+----------+-------+



In [33]:
training.printSchema()

root
 |-- Name: string (nullable = true)
 |-- Age: double (nullable = true)
 |-- Experience: double (nullable = true)
 |-- Salary: double (nullable = true)



In [34]:
training.columns

['Name', 'Age', 'Experience', 'Salary']

In [35]:
from pyspark.ml.feature import VectorAssembler

In [36]:
featureassembler = VectorAssembler(inputCols = ["Age","Experience"],outputCol = "Independent Features")

In [37]:
output = featureassembler.transform(training)

In [38]:
output.show()

+---------+----+----------+-------+--------------------+
|     Name| Age|Experience| Salary|Independent Features|
+---------+----+----------+-------+--------------------+
|    Krish|31.0|      10.0|30000.0|         [31.0,10.0]|
|Sudhanshu|30.0|       8.0|25000.0|          [30.0,8.0]|
|    Sunny|29.0|       4.0|20000.0|          [29.0,4.0]|
|     Paul|24.0|       3.0|20000.0|          [24.0,3.0]|
|   Harsha|21.0|       1.0|15000.0|          [21.0,1.0]|
|  Shubham|23.0|       2.0|18000.0|          [23.0,2.0]|
+---------+----+----------+-------+--------------------+



In [40]:
output.columns

['Name', 'Age', 'Experience', 'Salary', 'Independent Features']

In [42]:
finalized_data = output.select("Independent Features","Salary")

In [44]:
finalized_data.show()

+--------------------+-------+
|Independent Features| Salary|
+--------------------+-------+
|         [31.0,10.0]|30000.0|
|          [30.0,8.0]|25000.0|
|          [29.0,4.0]|20000.0|
|          [24.0,3.0]|20000.0|
|          [21.0,1.0]|15000.0|
|          [23.0,2.0]|18000.0|
+--------------------+-------+



In [47]:
from pyspark.ml.regression import LinearRegression
# Train test split
train_data,test_data =finalized_data.randomSplit([0.75,0.25])
regressor = LinearRegression(featuresCol = 'Independent Features', labelCol = 'Salary')
regressor = regressor.fit(train_data)

26/07/16 13:34:23 WARN Instrumentation: [37ff6ce0] regParam is zero, which might cause numerical instability and overfitting.
26/07/16 13:34:24 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/07/16 13:34:24 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK


In [48]:
regressor.coefficients

DenseVector([-64.8464, 1584.7554])

In [49]:
regressor.intercept

15414.10693970376

In [53]:
# Prediction 
pred_results = regressor.evaluate(test_data)

In [55]:
pred_results.predictions.show()

+--------------------+-------+------------------+
|Independent Features| Salary|        prediction|
+--------------------+-------+------------------+
|          [24.0,3.0]|20000.0|18612.059158134223|
+--------------------+-------+------------------+



In [56]:
pred_results.meanAbsoluteError,pred_results.meanSquaredError

(1387.9408418657767, 1926379.780519081)